In [3]:
# =========================
# 必要ライブラリ
# =========================
!pip install requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime, timedelta

# =========================
# 設定（自宅・会社）
# =========================
HOME = "東大宮"
WORK = "蕨"

# =========================
# ① 出社時間 → 家を出る時間
# =========================
def calc_depart_time(work_time_str, commute_minutes=30, buffer=10):
    """
    work_time_str : "09:00"
    commute_minutes : 通勤時間の仮定（分）
    buffer : 余裕時間
    """
    work_time = datetime.strptime(work_time_str, "%H:%M")
    depart_time = work_time - timedelta(minutes=commute_minutes + buffer)
    return depart_time.strftime("%H:%M")

# =========================
# ② Yahoo乗換案内URL生成
# =========================
def make_url(time_str):
    return f"https://transit.yahoo.co.jp/search/result?from={HOME}&to={WORK}&time={time_str}"

# =========================
# ③ 電車ルート取得
# =========================
def get_routes(time_str):
    url = make_url(time_str)
    headers = {"User-Agent": "Mozilla/5.0 (iPhone)"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")
    text = soup.text

    # 時刻・所要時間・乗換抽出
    times = re.findall(r"\d{2}:\d{2}→\d{2}:\d{2}", text)
    durations = re.findall(r"（\d+分）", text)
    transfers = re.findall(r"乗換:\d+回", text)

    routes = []
    for i in range(min(3, len(times))):
        minute = int(re.findall(r"\d+", durations[i])[0])
        routes.append({
            "time": times[i],
            "duration": minute,
            "transfer": transfers[i]
        })
    return routes

# =========================
# ④ ゆっくり分類
# =========================
def classify(minute):
    if minute >= 40:
        return "① ゆっくり"
    elif minute >= 30:
        return "② 少しゆっくり"
    else:
        return "③ 急ぐ"

# =========================
# ⑤ メインAI関数
# =========================
def commute_ai(work_time="09:00"):
    print("====== 通勤AI ======")

    # 家を出る時間計算
    depart_time = calc_depart_time(work_time)
    print(f"出社時間 : {work_time}")
    print(f"家を出る目安 : {depart_time}")
    print("--------------------")

    # 電車取得
    routes = get_routes(depart_time)

    print("おすすめ電車候補:")
    for i, r in enumerate(routes, 1):
        label = classify(r["duration"])
        print(f"{i}. {r['time']}（{r['duration']}分） {r['transfer']} → {label}")

    print("====================")

# =========================
# 実行（ここを変更）
# =========================
commute_ai("09:00")   # ← 9時出勤



====== 通勤AI ======
出社時間 : 09:00
家を出る目安 : 08:20
--------------------
おすすめ電車候補:
1. 23:05→23:30（25分） 乗換:1回 → ③ 急ぐ
2. 23:05→23:37（32分） 乗換:1回 → ② 少しゆっくり
3. 23:29→23:54（25分） 乗換:1回 → ③ 急ぐ


In [ ]:
pip install selenium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from datetime import datetime
import re
import time

HOME_LEAVE_TIME = "08:20"

def str_to_time(t):
    return datetime.strptime(t, "%H:%M")

leave_time = str_to_time(HOME_LEAVE_TIME)

driver = webdriver.Chrome()
driver.get("https://transit.yahoo.co.jp/search/result?from=東大宮&to=蕨")
time.sleep(5)

html = driver.page_source
driver.quit()

# 時刻抽出
routes = re.findall(r'(\d{1,2}:\d{2})→(\d{1,2}:\d{2}).+?乗換:(\d+)回', html)
transfers = re.findall(r'(\d{1,2}:\d{2})着(\d{1,2}:\d{2})発', html)

print("====== 通勤AI ======")
print("出社時間 : 09:00")
print(f"家を出る目安 : {HOME_LEAVE_TIME}")
print("--------------------")
print("おすすめ電車候補:")

rank = 1
for dep, arr, trans in routes:
    dep_time = str_to_time(dep)
    if dep_time < leave_time:
        continue

    print(f"{rank}. {dep}→{arr}（乗換:{trans}回）")

    if transfers:
        arr_t, dep_t = transfers[0]
        margin = (str_to_time(dep_t) - str_to_time(arr_t)).seconds // 60
        print(f"   乗換余裕: {margin}分（{arr_t}着 → {dep_t}発）")

    rank += 1



SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5ad1f320cd0a <unknown>
#1 0x5ad1f2c1c682 <unknown>
#2 0x5ad1f2c5a23b <unknown>
#3 0x5ad1f2c54f09 <unknown>
#4 0x5ad1f2ca68de <unknown>
#5 0x5ad1f2ca5fcc <unknown>
#6 0x5ad1f2c6388f <unknown>
#7 0x5ad1f2c64651 <unknown>
#8 0x5ad1f31d1159 <unknown>
#9 0x5ad1f31d4061 <unknown>
#10 0x5ad1f31bd919 <unknown>
#11 0x5ad1f31d4c2e <unknown>
#12 0x5ad1f31a3c90 <unknown>
#13 0x5ad1f31f9358 <unknown>
#14 0x5ad1f31f9528 <unknown>
#15 0x5ad1f320b353 <unknown>
#16 0x7a133e8feac3 <unknown>


In [ ]:
!pip install selenium
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,728 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Get:

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

service = Service("/usr/lib/chromium-browser/chromedriver")

driver = webdriver.Chrome(service=service, options=chrome_options)

driver.get("https://transit.yahoo.co.jp/")
print(driver.title)

driver.quit()


WebDriverException: Message: Service /usr/lib/chromium-browser/chromedriver unexpectedly exited. Status code was: 1


In [ ]:
import requests
import datetime

FROM = "東大宮"
TO = "蕨"
HOME_LEAVE = "08:20"  # 家を出る時間

def yahoo_search(time_str):
    t = datetime.datetime.strptime(time_str, "%H:%M")
    time_param = t.strftime("%H%M")

    url = f"https://transit.yahoo.co.jp/search/result?from={FROM}&to={TO}&time={time_param}"
    return url

print("検索URL:", yahoo_search(HOME_LEAVE))


検索URL: https://transit.yahoo.co.jp/search/result?from=東大宮&to=蕨&time=0820


In [ ]:
import requests
from bs4 import BeautifulSoup

url = yahoo_search("08:20")
html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)


東大宮から蕨への乗換案内 - Yahoo!路線情報


In [ ]:
routes = soup.select("div.routeSummary")
for r in routes:
    print(r.text.strip())
    print("------")


In [ ]:
routes = soup.select("div.routeSummary")
print(len(routes))


0


In [ ]:
# ===== 設定 =====
FROM = "東大宮"
TO = "蕨"
HOME_LEAVE_TIME = "09:00"   # 家を出る時間
SHOW_NUM = 10

# =========================
import requests
from bs4 import BeautifulSoup
import datetime
import re

def str_to_time(t):
    return datetime.datetime.strptime(t, "%H:%M")

def yahoo_search_mobile(time_str):
    t = str_to_time(time_str)
    time_param = t.strftime("%H%M")
    url = f"https://transit.yahoo.co.jp/m/search/result?from={FROM}&to={TO}&time={time_param}"
    headers = {"User-Agent": "Mozilla/5.0 (iPhone)"}
    html = requests.get(url, headers=headers).text
    return BeautifulSoup(html, "html.parser")

# 時間差分（分）
def diff_minutes(t1, t2):
    f = str_to_time(t1)
    t = str_to_time(t2)
    return int((t - f).total_seconds() / 60)

# 余裕判定
def judge_transfer(mins):
    if mins <= 3:
        return "①急いで"
    elif mins <= 7:
        return "②少しゆっくり"
    else:
        return "②ゆっくり"

# ルート解析（超強化）
def parse_routes(soup):
    routes = []

    for route in soup.select("section.route"):
        text = route.text.replace("\n", " ")

        # 出発・到着
        m = re.search(r"(\d{2}:\d{2})発.*?(\d{2}:\d{2})着", text)
        if not m:
            continue

        first_dep, final_arr = m.group(1), m.group(2)

        # 駅名と時刻抽出
        times = re.findall(r"(\d{2}:\d{2})", text)
        stations = re.findall(r"[^\s]+駅", text)

        # 乗換駅推定（最初の到着駅）
        if len(times) >= 3:
            first_arr = times[1]
            change_dep = times[2]
        else:
            continue

        # 駅名抽出（大宮など）
        station_match = re.search(r"着([^\s]+)駅", text)
        change_station = station_match.group(1) if station_match else "乗換駅"

        routes.append({
            "first_dep": first_dep,
            "first_arr": first_arr,
            "change_station": change_station,
            "change_dep": change_dep,
            "final_arr": final_arr
        })
    return routes

# =========================
# 実行
soup = yahoo_search_mobile(HOME_LEAVE_TIME)
routes = parse_routes(soup)

print("====== 通勤AI ======")
print("家を出る時間 :", HOME_LEAVE_TIME)
print("--------------------")

# 同じ1本目ごとにグループ化
grouped = {}
for r in routes:
    key = (r["first_dep"], r["first_arr"], r["change_station"])
    grouped.setdefault(key, []).append(r)

# 出力
idx = 1
for key, group in list(grouped.items())[:SHOW_NUM]:
    first_dep, first_arr, station = key
    print(f"候補{idx}")

    for r in group:
        transfer_min = diff_minutes(first_arr, r["change_dep"])
        level = judge_transfer(transfer_min)
        print(f" {first_dep}→{first_arr}{station}着  {r['change_dep']}{station}発→{r['final_arr']}（乗換{transfer_min}分）{level}")

    print()
    idx += 1


====== 通勤AI ======
家を出る時間 : 09:00
--------------------


In [ ]:
soup = yahoo_search_mobile("09:00")

print(len(soup.select("section.route")))


0


In [6]:
# ====== 設定 ======
FROM = "東大宮"
TO = "蕨"
HOME_LEAVE_TIME = "09:00"   # 家を出る時間
SHOW_NUM = 5

# ==================
import requests
from bs4 import BeautifulSoup
import re
import datetime

# 時刻変換
def str_to_time(t):
    return datetime.datetime.strptime(t, "%H:%M")

# 時刻差（分）
def diff_minutes(t1, t2):
    return int((str_to_time(t2) - str_to_time(t1)).total_seconds() / 60)

# 乗換余裕判定
def judge_transfer(mins):
    if mins <= 3:
        return "①急いで"
    elif mins <= 7:
        return "②少しゆっくり"
    else:
        return "②ゆっくり"

# Yahoo検索URL（PC軽量HTML・確実取得）
def yahoo_url(time_str):
    t = str_to_time(time_str).strftime("%H%M")
    return f"https://transit.yahoo.co.jp/search/result?from={FROM}&to={TO}&time={t}&al=1&ic=1&shin=1&ex=1"

# ====== データ取得 ======
url = yahoo_url(HOME_LEAVE_TIME)
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

# ====== ルート解析 ======
routes = []

for r in soup.select("div.routeSummary"):
    text = r.get_text(" ", strip=True)
    times = re.findall(r"\d{2}:\d{2}", text)

    # 東大宮→大宮→蕨 の最低構成
    if len(times) >= 4:
        first_dep = times[0]
        first_arr = times[1]
        change_dep = times[2]
        final_arr = times[-1]

        routes.append({
            "first_dep": first_dep,
            "first_arr": first_arr,
            "change_station": "大宮",   # 今回は固定（本番はAPI推奨）
            "change_dep": change_dep,
            "final_arr": final_arr
        })

# ====== 同じ1本目でグループ化 ======
grouped = {}
for r in routes:
    key = (r["first_dep"], r["first_arr"])
    grouped.setdefault(key, []).append(r)

# ====== 出力 ======
print("====== 通勤AI ======")
print("家を出る時間 :", HOME_LEAVE_TIME)
print("--------------------")

idx = 1
for key, group in list(grouped.items())[:SHOW_NUM]:
    first_dep, first_arr = key

    for r in group:
        change_dep = r["change_dep"]
        final_arr = r["final_arr"]
        station = r["change_station"]

        transfer = diff_minutes(first_arr, change_dep)
        level = judge_transfer(transfer)

        print(f"候補{idx} {first_dep}→{first_arr}{station}着 {change_dep}{station}発→{final_arr}（乗換{transfer}分）{level}")

    idx += 1


====== 通勤AI ======
家を出る時間 : 09:00
--------------------


In [ ]:
from datetime import datetime, timedelta

# ========================
# 設定
# ========================
LEAVE_HOME_TIME = "09:00"  # 家を出る時間
TRANSFER_LIMIT_MIN = 10    # 乗り換え猶予分（最大）

# ========================
# ダミー時刻表（実運用はAPI・CSVに置き換える）
# ========================

# 自宅最寄り → 大宮
train_to_omiya = [
    "09:05", "09:20", "09:37", "09:55"
]

# 大宮 → 東京
omiya_to_tokyo = [
    "09:26", "09:29", "09:34", "09:42", "09:49", "10:05"
]

# ========================
# 時刻処理関数
# ========================
def str_to_time(t):
    return datetime.strptime(t, "%H:%M")

def time_diff_min(t1, t2):
    return int((t2 - t1).total_seconds() / 60)

# ========================
# 候補探索
# ========================
def find_commute_candidates():
    leave_time = str_to_time(LEAVE_HOME_TIME)
    results = []

    for first in train_to_omiya:
        first_time = str_to_time(first)

        # 家を出た後の電車のみ
        if first_time < leave_time:
            continue

        # 大宮到着（仮に6分）
        omiya_arrive = first_time + timedelta(minutes=6)

        # 大宮乗り換え候補探す
        for second in omiya_to_tokyo:
            second_time = str_to_time(second)

            transfer_min = time_diff_min(omiya_arrive, second_time)

            # 乗り換え条件
            if 0 <= transfer_min <= TRANSFER_LIMIT_MIN:
                arrive_tokyo = second_time + timedelta(minutes=15)  # 仮: 15分

                results.append({
                    "first": first,
                    "omiya_arrive": omiya_arrive.strftime("%H:%M"),
                    "second": second,
                    "tokyo_arrive": arrive_tokyo.strftime("%H:%M"),
                    "transfer": transfer_min
                })

    return results

# ========================
# 出力
# ========================
def print_result():
    print("====== 通勤AI ======")
    print(f"家を出る時間 : {LEAVE_HOME_TIME}")
    print("--------------------")

    candidates = find_commute_candidates()

    if not candidates:
        print("電車候補なし")
        return

    for i, c in enumerate(candidates, 1):
        level = "急いで" if c["transfer"] <= 3 else "ゆっくり"
        print(f"候補{i} {c['first']}→{c['omiya_arrive']}大宮着  "
              f"{c['second']}大宮発→{c['tokyo_arrive']}東京着 "
              f"(乗り換え{c['transfer']}分) {level}")

# ========================
# 実行
# ========================
if __name__ == "__main__":
    print_result()


====== 通勤AI ======
家を出る時間 : 09:00
--------------------
候補1 09:20→09:26大宮着  09:26大宮発→09:41東京着 (乗り換え0分) 急いで
候補2 09:20→09:26大宮着  09:29大宮発→09:44東京着 (乗り換え3分) 急いで
候補3 09:20→09:26大宮着  09:34大宮発→09:49東京着 (乗り換え8分) ゆっくり
候補4 09:37→09:43大宮着  09:49大宮発→10:04東京着 (乗り換え6分) ゆっくり
候補5 09:55→10:01大宮着  10:05大宮発→10:20東京着 (乗り換え4分) ゆっくり


In [4]:
# =========================
# 必要ライブラリ
# =========================
!pip install requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

# =========================
# 設定（駅名）
# =========================
HOME = "東大宮"
WORK = "さいたま新都心"

# =========================
# ① Yahoo乗換案内URL生成
# =========================
def make_url(from_station, to_station, time_str):
    return f"https://transit.yahoo.co.jp/search/result?from={from_station}&to={to_station}&time={time_str}"

# =========================
# ② 電車ルート取得
# =========================
def get_routes(from_station, to_station, time_str):
    url = make_url(from_station, to_station, time_str)
    headers = {"User-Agent": "Mozilla/5.0 (iPhone)"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")
    text = soup.text

    # 時刻・所要時間・乗換抽出
    times = re.findall(r"\d{2}:\d{2}→\d{2}:\d{2}", text)
    durations = re.findall(r"（\d+分）", text)
    transfers = re.findall(r"乗換:\d+回", text)

    routes = []
    for i in range(min(3, len(times))):
        minute = int(re.findall(r"\d+", durations[i])[0])
        routes.append({
            "time": times[i],
            "duration": minute,
            "transfer": transfers[i]
        })
    return routes

# =========================
# ③ ゆっくり分類
# =========================
def classify(minute):
    if minute >= 40:
        return "① ゆっくり"
    elif minute >= 30:
        return "② 少しゆっくり"
    else:
        return "③ 急ぐ"

# =========================
# ④ メインAI（行き・帰り）
# =========================
def commute_ai(search_time="08:00"):
    print("====== 通勤AI ======")
    print(f"検索時刻 : {search_time}")
    print("--------------------")

    # 行き
    print(f"【行き】{HOME} → {WORK}")
    routes_go = get_routes(HOME, WORK, search_time)
    for i, r in enumerate(routes_go, 1):
        label = classify(r["duration"])
        print(f"{i}. {r['time']}（{r['duration']}分） {r['transfer']} → {label}")

    print("--------------------")

    # 帰り
    print(f"【帰り】{WORK} → {HOME}")
    routes_back = get_routes(WORK, HOME, search_time)
    for i, r in enumerate(routes_back, 1):
        label = classify(r["duration"])
        print(f"{i}. {r['time']}（{r['duration']}分） {r['transfer']} → {label}")

    print("====================")

# =========================
# 実行
# =========================
commute_ai("08:10")   # ← 検索したい時刻を自由に指定


====== 通勤AI ======
検索時刻 : 08:10
--------------------
【行き】東大宮 → さいたま新都心
1. 23:29→23:38（9分） 乗換:0回 → ③ 急ぐ
2. 23:51→00:18（27分） 乗換:1回 → ③ 急ぐ
3. 23:29→23:38（9分） 乗換:0回 → ③ 急ぐ
--------------------
【帰り】さいたま新都心 → 東大宮
1. 23:09→23:18（9分） 乗換:0回 → ③ 急ぐ
2. 23:26→23:39（13分） 乗換:1回 → ③ 急ぐ
3. 23:17→23:39（22分） 乗換:1回 → ③ 急ぐ


In [5]:
import requests
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime

HOME = "東大宮"
WORK = "さいたま新都心"

# =========================
# URL生成
# =========================
def make_url(frm, to, time_str):
    return f"https://transit.yahoo.co.jp/search/result?from={frm}&to={to}&time={time_str}"

# =========================
# 乗換判定
# =========================
def classify_transfer(minute):
    if minute == 0:
        return ""
    elif minute >= 5:
        return "ゆっくり"
    else:
        return "急いで"

# =========================
# 時刻差計算
# =========================
def diff_minutes(t1, t2):
    fmt = "%H:%M"
    a = datetime.strptime(t1, fmt)
    b = datetime.strptime(t2, fmt)
    return int((b - a).seconds / 60)

# =========================
# Yahoo詳細JSON抽出
# =========================
def get_yahoo_routes(time_str):
    url = make_url(HOME, WORK, time_str)
    headers = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")

    # JSONが埋め込まれているscriptを探す
    scripts = soup.find_all("script")
    json_text = None

    for s in scripts:
        if s.string and "window.__INITIAL_STATE__" in s.string:
            json_text = s.string
            break

    if not json_text:
        print("JSONデータ取得失敗")
        return []

    # JSON部分だけ抽出
    json_str = re.search(r"__INITIAL_STATE__ = (.*);", json_text).group(1)
    data = json.loads(json_str)

    routes = data["transit"]["search"]["results"]

    return routes

# =========================
# メイン
# =========================
def commute_ai(time_str="09:00"):
    routes = get_yahoo_routes(time_str)

    print("====== 通勤AI ======")

    for idx, r in enumerate(routes[:4], 1):
        sections = r["sections"]

        # 直通
        if len(sections) == 1:
            s = sections[0]
            print(f"候補{idx} {s['fromTime']}→{s['toTime']} (乗り換え0分)")
            continue

        # 乗換あり（1回想定）
        s1 = sections[0]
        s2 = sections[1]

        arrive = s1["toTime"]
        depart = s2["fromTime"]
        transfer = diff_minutes(arrive, depart)
        label = classify_transfer(transfer)

        print(
            f"候補{idx} {s1['fromTime']}→{arrive}{s1['to']}着  "
            f"{depart}{s2['from']}発→{s2['toTime']}{s2['to']}着 "
            f"(乗り換え{transfer}分) {label}"
        )

# 実行
commute_ai("09:15")


JSONデータ取得失敗
====== 通勤AI ======


In [7]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

HOME = "東大宮"
WORK = "さいたま新都心"

# =========================
# URL生成
# =========================
def make_url(time_str):
    return f"https://transit.yahoo.co.jp/search/result?from={HOME}&to={WORK}&time={time_str}"

# =========================
# 時刻差計算
# =========================
def diff_minutes(t1, t2):
    fmt = "%H:%M"
    a = datetime.strptime(t1, fmt)
    b = datetime.strptime(t2, fmt)
    return int((b - a).seconds / 60)

# =========================
# Yahooスクレイピング
# =========================
def get_transfer_times(time_str="09:00"):
    url = make_url(time_str)
    headers = {"User-Agent": "Mozilla/5.0 (iPhone)"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")
    text = soup.text

    # 全時刻抽出
    all_times = re.findall(r"\d{2}:\d{2}", text)

    transfers = []

    # 連続する時刻を仮に「到着→出発」とみなす（簡易AI）
    for i in range(0, len(all_times)-1, 2):
        arrive = all_times[i]
        depart = all_times[i+1]
        diff = diff_minutes(arrive, depart)

        if diff <= 20:  # 乗換っぽいものだけ
            transfers.append((arrive, depart, diff))

    return transfers[:5]

# =========================
# 実行
# =========================
def commute_ai(time_str="09:15"):
    transfers = get_transfer_times(time_str)

    print("===== 乗換時間AI =====")
    for i, (a, d, diff) in enumerate(transfers, 1):
        label = "ゆっくり" if diff >= 5 else "急いで"
        print(f"候補{i} {a}着 → {d}発 (乗り換え{diff}分) {label}")

commute_ai("09:15")


===== 乗換時間AI =====
候補1 23:34着 → 23:51発 (乗り換え17分) ゆっくり
候補2 23:53着 → 23:57発 (乗り換え4分) 急いで
候補3 00:15着 → 00:18発 (乗り換え3分) 急いで


In [1]:
# =========================
# 必要ライブラリ
# =========================
!pip install selenium beautifulsoup4

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import quote

# =========================
# 駅設定
# =========================
HOME = "東大宮"
WORK = "さいたま新都心"

# =========================
# URL作成
# =========================
def make_url(time_str=None):
    f = quote(HOME)
    t = quote(WORK)

    if time_str:
        return f"https://transit.yahoo.co.jp/search/result?from={f}&to={t}&time={time_str}"
    else:
        return f"https://transit.yahoo.co.jp/search/result?from={f}&to={t}"

# =========================
# Selenium起動
# =========================
def create_driver():
    options = Options()
    options.add_argument("--headless")  # 画面表示したい場合は削除
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1200,800")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )

    return webdriver.Chrome(options=options)

# =========================
# 電車候補取得
# =========================
def get_routes(time_str=None):
    url = make_url(time_str)
    print("アクセス:", url)

    driver = create_driver()
    driver.get(url)
    time.sleep(5)  # ページロード待ち

    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text()

    # 時刻抽出
    times = re.findall(r"\d{2}:\d{2}→\d{2}:\d{2}", text)

    # 所要時間
    durations = re.findall(r"（\d+分）", text)

    # 乗換回数
    transfers = re.findall(r"乗換:\d+回", text)

    routes = []
    for i in range(min(5, len(times))):
        minute = int(re.findall(r"\d+", durations[i])[0])
        transfer = transfers[i] if i < len(transfers) else "乗換:0回"

        routes.append({
            "time": times[i],
            "duration": minute,
            "transfer": transfer
        })
    return routes

# =========================
# 表示
# =========================
def commute_ai(time_str=None):
    print("====== 通勤AI ======")
    routes = get_routes(time_str)

    print("電車候補:")
    for i, r in enumerate(routes, 1):
        print(f"{i}. {r['time']}（{r['duration']}分） {r['transfer']}")

    print("====================")


# =========================
# 実行
# =========================
commute_ai("07:30")   # 時刻指定（省略可）


====== 通勤AI ======
アクセス: https://transit.yahoo.co.jp/search/result?from=%E6%9D%B1%E5%A4%A7%E5%AE%AE&to=%E3%81%95%E3%81%84%E3%81%9F%E3%81%BE%E6%96%B0%E9%83%BD%E5%BF%83&time=07:30


SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5b660889ad0a <unknown>
#1 0x5b66082aa682 <unknown>
#2 0x5b66082e823b <unknown>
#3 0x5b66082e2f09 <unknown>
#4 0x5b66083348de <unknown>
#5 0x5b6608333fcc <unknown>
#6 0x5b66082f188f <unknown>
#7 0x5b66082f2651 <unknown>
#8 0x5b660885f159 <unknown>
#9 0x5b6608862061 <unknown>
#10 0x5b660884b919 <unknown>
#11 0x5b6608862c2e <unknown>
#12 0x5b6608831c90 <unknown>
#13 0x5b6608887358 <unknown>
#14 0x5b6608887528 <unknown>
#15 0x5b6608899353 <unknown>
#16 0x79ffc8ad6ac3 <unknown>


In [2]:
!pip install requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import quote

HOME = "東大宮"
WORK = "さいたま新都心"

def make_url(time_str="07:30"):
    f = quote(HOME)
    t = quote(WORK)
    return f"https://transit.yahoo.co.jp/search/result?from={f}&to={t}&time={time_str}"

def get_routes(time_str="07:30"):
    url = make_url(time_str)
    headers = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text()

    times = re.findall(r"\d{2}:\d{2}→\d{2}:\d{2}", text)
    durations = re.findall(r"（\d+分）", text)
    transfers = re.findall(r"乗換:\d+回", text)

    routes = []
    for i in range(min(5, len(times))):
        minute = int(re.findall(r"\d+", durations[i])[0])
        transfer = transfers[i] if i < len(transfers) else "乗換:0回"
        routes.append((times[i], minute, transfer))

    return routes

def commute_ai():
    print("====== 通勤AI ======")
    for i, r in enumerate(get_routes(), 1):
        print(f"{i}. {r[0]}（{r[1]}分） {r[2]}")
    print("====================")

commute_ai()


====== 通勤AI ======


IndexError: list index out of range

In [3]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import quote

HOME = "東大宮"
WORK = "さいたま新都心"

def make_url(time_str="07:30"):
    f = quote(HOME)
    t = quote(WORK)
    return f"https://transit.yahoo.co.jp/search/result?from={f}&to={t}&time={time_str}"

def get_routes(time_str="07:30"):
    url = make_url(time_str)
    headers = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text()

    times = re.findall(r"\d{2}:\d{2}→\d{2}:\d{2}", text)
    durations = re.findall(r"（\d+分）", text)
    transfers = re.findall(r"乗換:\d+回", text)

    routes = []
    for i in range(min(5, len(times))):
        # 所要時間が無い場合は None
        if i < len(durations):
            minute = int(re.findall(r"\d+", durations[i])[0])
        else:
            minute = None

        # 乗換情報が無い場合は 0回
        transfer = transfers[i] if i < len(transfers) else "乗換:0回"

        routes.append({
            "time": times[i],
            "duration": minute,
            "transfer": transfer
        })
    return routes


def commute_ai():
    print("====== 通勤AI ======")
    routes = get_routes()
    print("電車候補:")
    for i, r in enumerate(routes, 1):
        print(f"{i}. {r['time']}（{r['duration']}分） {r['transfer']}")
    print("====================")

commute_ai()


====== 通勤AI ======
電車候補:
1. 05:47→05:56（None分） 乗換:0回
2. 06:03→06:12（None分） 乗換:0回
3. 06:14→06:24（None分） 乗換:0回


In [4]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import quote
from datetime import datetime

HOME = "東大宮"
WORK = "さいたま新都心"

def make_url(time_str="06:30"):
    return f"https://transit.yahoo.co.jp/search/result?from={quote(HOME)}&to={quote(WORK)}&time={time_str}"

def time_diff(a, b):
    fmt = "%H:%M"
    return int((datetime.strptime(b, fmt) - datetime.strptime(a, fmt)).seconds / 60)

def classify_transfer(minute):
    if minute >= 5:
        return "ゆっくり"
    else:
        return "急いで"

def get_routes(time_str="06:30"):
    html = requests.get(make_url(time_str), headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text()

    # 全時刻抽出
    times = re.findall(r"\d{2}:\d{2}", text)

    routes = []
    # 2個ずつペアにして仮ルート生成（簡易版）
    for i in range(0, min(10, len(times)-1), 2):
        start = times[i]
        end = times[i+1]
        duration = time_diff(start, end)
        routes.append((start, end, duration))

    return routes


def commute_ai():
    print("電車候補:")
    routes = get_routes()

    for i, (s, e, d) in enumerate(routes, 1):
        # 仮の乗換時間（デモ用）
        transfer_min = d % 10   # ← 本来はDOM解析必要
        label = classify_transfer(transfer_min)

        print(f"{i}. {s}→{e}（{transfer_min}分） 乗換:1回 → {label}")

commute_ai()


電車候補:
1. 00:08→05:47（9分） 乗換:1回 → ゆっくり
2. 05:56→06:03（7分） 乗換:1回 → ゆっくり
3. 06:12→06:14（2分） 乗換:1回 → 急いで
4. 06:24→05:47（3分） 乗換:1回 → 急いで
5. 05:56→05:47（1分） 乗換:1回 → 急いで
